# 11-1. 라우터 정확도

선택 심화. `11`을 마친 뒤 본다.

`11`의 라우터는 질문 여섯 개로 확인했다. 여기서는 질문을 예순 개로 늘리고 라우터 네 종류를 같은 질문에 돌린다. **규칙, 임베딩, 하위 모델, 상위 모델**이다.
틀린 방향을 혼동 행렬로 보고, 라우터가 덜 확실할 때 다른 길로 보내는 것이 얼마나 도움이 되는지 잰다.

### 공통 설정

`11`과 같은 블록이다. 임베딩 모델을 더 쓴다. 질문을 스레드로 동시에 보내므로 429 에 대비해 클라이언트의 재시도 횟수를 늘린다.
임베딩 응답의 usage 는 모양이 달라서 `keep_embed`로 따로 담는다.

In [1]:
import json
import math
import os
import re
import time
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
from types import SimpleNamespace
from typing import Literal
import numpy as np
from dotenv import load_dotenv, find_dotenv
from openai import OpenAI
from pydantic import BaseModel

load_dotenv(find_dotenv(".env", usecwd=True), override=True)
MODEL = os.environ["OPENAI_DEFAULT_MODEL"]
LOWER = os.environ["OPENAI_LOWER_MODEL"]
EMBEDDING = os.environ["OPENAI_EMBEDDING_MODEL"]
client = OpenAI(max_retries=6)

PRICES = {MODEL: (2.50, 15.00), LOWER: (0.75, 4.50), EMBEDDING: (0.02, 0.0)}
USAGE = []


def keep(resp):
    USAGE.append((resp.model, resp.usage))
    return resp


def keep_embed(resp):
    USAGE.append((resp.model, SimpleNamespace(input_tokens=resp.usage.prompt_tokens, output_tokens=0)))
    return resp


def dollars(rows):
    total = 0.0
    for name, u in rows:
        p = next((PRICES[k] for k in sorted(PRICES, key=len, reverse=True) if name.startswith(k)), None)
        if p:
            total += u.input_tokens / 1e6 * p[0] + u.output_tokens / 1e6 * p[1]
    return total

### 전문가와 도구

`11`의 세 전문가를 그대로 쓴다. 전문가는 `CONFIG`를 물려받은 설정이다. 라우터 평가에는 전문가를 돌리지 않는다. 뒤의 대체 경로 절에서만 돌린다.

In [2]:
CORPUS = Path("../../data/corpus/graph")
TITLES = sorted(f.stem for f in CORPUS.glob("*.md"))


def grep_docs(pattern):
    return [{"title": f.stem, "count": f.read_text(encoding="utf-8").count(pattern)}
            for f in sorted(CORPUS.glob("*.md")) if pattern in f.read_text(encoding="utf-8")]


def read_doc(title):
    f = CORPUS / f"{title}.md"
    if not f.exists():
        raise FileNotFoundError(f"'{title}' 문서가 없다")
    return f.read_text(encoding="utf-8")[:4000]


def spec(name, desc, props):
    return {"type": "function", "name": name, "description": desc, "strict": True,
            "parameters": {"type": "object", "properties": props,
                           "required": list(props), "additionalProperties": False}}


GREP = spec("grep_docs", "낱말이 든 문서의 제목과 등장 횟수를 돌려준다.",
            {"pattern": {"type": "string", "description": "찾을 낱말"}})
READ = spec("read_doc", "문서 한 편의 앞부분을 돌려준다.",
            {"title": {"type": "string", "description": "문서 제목"}})
IMPL = {"grep_docs": grep_docs, "read_doc": read_doc}
CONFIG = {"model": MODEL, "instructions": "도구로 확인한 것만 답한다.", "tools": [GREP, READ], "impl": IMPL,
          "max_turns": 5, "max_tokens": 200000, "max_seconds": 180, "gated": [],
          "gate_error": "error: 승인이 필요하다.", "tool_error": "error: {e}"}

AGENTS = {
    "counter": {**CONFIG, "tools": [GREP],
                "instructions": "세는 질문만 맡는다. grep_docs 로 세고 숫자로 답한다."},
    "reader": {**CONFIG, "tools": [READ],
               "instructions": "한 문서의 내용을 묻는 질문만 맡는다. read_doc 으로 읽고 답한다."},
    "refuser": {**CONFIG, "tools": [],
                "instructions": "이 코퍼스는 대한민국 임시정부 문서 50편이다."
                                " 범위 밖 질문에는 답할 수 없다고만 말한다."},
}
NAMES = list(AGENTS)

### 질문 셋

정답 담당자를 붙인 질문을 두 묶음 만든다. 셈은 counter, 한 문서의 내용은 reader, 코퍼스 밖은 refuser 다.

`DEV`는 라우터를 만들면서 본 질문 마흔 개다. 절반은 분명한 질문이고, 나머지는 다른 담당자 쪽으로 끌리는 말을 일부러 넣은 경계 질문이다.
"김구 문서에 광복군이 몇 번" 은 문서 이름이 있지만 세는 질문이고, "김구의 생년은 몇 년" 은 숫자를 묻지만 읽는 질문이다. "임시정부 청사 입장료" 는 주제가 가깝지만 코퍼스 밖이다.

`HELD`는 라우터를 다 만든 뒤에 다른 말투로 새로 쓴 스물한 개다. 라우터를 고칠 때 보지 않는다.

In [3]:
DEV = [
    ("counter", "분명", "한국독립당은 몇 편의 문서에 나오는가"),
    ("counter", "분명", "광복군이 가장 많이 나오는 문서는 무엇인가"),
    ("counter", "분명", "의열단이라는 말이 들어간 문서 수를 센다"),
    ("counter", "분명", "하와이가 언급된 문서는 모두 몇 개인가"),
    ("counter", "분명", "상하이가 가장 자주 나오는 문서 세 개를 꼽는다"),
    ("counter", "분명", "신민회가 나오는 문서가 다섯 편이 넘는가"),
    ("counter", "분명", "독립신문이 등장하는 문서를 전부 나열한다"),
    ("counter", "분명", "충칭과 난징 중 어느 쪽이 더 많은 문서에 나오는가"),
    ("reader", "분명", "윤봉길 문서에 적힌 의거 날짜는 언제인가"),
    ("reader", "분명", "한인애국단 문서는 이 단체를 어떻게 설명하는가"),
    ("reader", "분명", "안중근 문서의 첫 문단을 요약한다"),
    ("reader", "분명", "서재필 문서에 따르면 그는 어디서 태어났는가"),
    ("reader", "분명", "자유시 참변 문서는 사건의 원인을 무엇이라 하는가"),
    ("reader", "분명", "이회영 문서는 그의 형제들에 대해 무엇이라 하는가"),
    ("reader", "분명", "연통제 문서는 이 제도를 어떻게 정의하는가"),
    ("reader", "분명", "유관순 문서의 첫 문장은 무엇인가"),
    ("refuser", "분명", "오늘 서울 날씨는 어떤가"),
    ("refuser", "분명", "파이썬으로 리스트를 정렬하는 법을 알려 달라"),
    ("refuser", "분명", "비트코인 시세를 알려 달라"),
    ("refuser", "분명", "다음 주 회의 일정을 잡아 달라"),
    ("refuser", "분명", "김치찌개 끓이는 법을 알려 달라"),
    ("refuser", "분명", "영어 이메일 초안을 써 달라"),
    ("refuser", "분명", "2026년 월드컵 우승 팀은 어디인가"),
    ("refuser", "분명", "엑셀에서 피벗 테이블 만드는 법"),
    ("counter", "경계", "김구 문서에 광복군이라는 말이 몇 번 나오는가"),
    ("counter", "경계", "이봉창 의거 문서와 윤봉길 문서 중 어느 쪽이 도쿄를 더 많이 언급하는가"),
    ("counter", "경계", "김원봉을 다룬 문서가 몇 편인가"),
    ("counter", "경계", "안중근이 언급된 문서들의 제목을 알려 달라"),
    ("counter", "경계", "이 문서들에 가장 자주 나오는 독립운동가는 김구인가 이승만인가"),
    ("reader", "경계", "김구 문서에 적힌 김구의 생년은 몇 년인가"),
    ("reader", "경계", "이승만 문서는 그가 몇 년 동안 대통령이었다고 하는가"),
    ("reader", "경계", "국민대표회의 문서에 참가자가 몇 명이었다고 나오는가"),
    ("reader", "경계", "백정기 문서에 적힌 그의 사망 연도는 언제인가"),
    ("reader", "경계", "임시정부는 언제 세워졌는가"),
    ("reader", "경계", "신채호는 어떤 사람인가"),
    ("reader", "경계", "한국친우회가 무엇인지 설명해 달라"),
    ("refuser", "경계", "임시정부 청사의 지금 입장료는 얼마인가"),
    ("refuser", "경계", "김구 기념관 가는 길을 알려 달라"),
    ("refuser", "경계", "안중근 유해 발굴의 최신 소식은 무엇인가"),
    ("refuser", "경계", "한국 독립운동사 입문서를 추천해 달라"),
]

HELD = [
    ("counter", "새", "광복군은 문서 몇 군데에서 언급되는가"),
    ("counter", "새", "임시의정원 언급이 제일 많은 글은"),
    ("counter", "새", "조선민족혁명당 이야기가 나오는 글이 열 편을 넘나"),
    ("counter", "새", "워싱턴이라는 지명이 들어간 문서를 모두 찾아 달라"),
    ("counter", "새", "신한청년당과 국민대표회의 가운데 더 널리 언급되는 쪽은"),
    ("counter", "새", "이승만이라는 이름은 전체에서 몇 차례 나오나"),
    ("counter", "새", "대한매일신보를 언급하는 글의 목록"),
    ("reader", "새", "강우규가 1919년에 무슨 일을 했는지 알려 달라"),
    ("reader", "새", "박은식의 대표 저서는 무엇인가"),
    ("reader", "새", "독립공채표가 무엇이었는지 설명해 달라"),
    ("reader", "새", "백의사는 어떤 조직인가"),
    ("reader", "새", "안재홍은 해방 후 어떤 활동을 했나"),
    ("reader", "새", "남궁억이 세운 학교의 이름은"),
    ("reader", "새", "필라델피아 한인자유대회의 결의 내용은"),
    ("refuser", "새", "내일 우산이 필요할까"),
    ("refuser", "새", "SQL 조인 종류를 설명해 달라"),
    ("refuser", "새", "독립기념관 주차 요금은 얼마인가"),
    ("refuser", "새", "광복절 연휴에 갈 만한 여행지 추천"),
    ("refuser", "새", "이 문장을 영어로 번역해 달라: 안녕하세요"),
    ("refuser", "새", "요즘 유행하는 역사 드라마는 무엇인가"),
    ("refuser", "새", "회의록 요약 양식을 만들어 달라"),
]
ALL = DEV + HELD
print(len(DEV), len(HELD), Counter(g for g, _, _ in ALL))

40 21 Counter({'reader': 22, 'counter': 20, 'refuser': 19})


### 채점

정답률은 묶음별로 따로 낸다. 혼동 행렬은 행이 정답, 열이 라우터가 고른 담당자다. 대각선 밖이 오류의 방향이다.

In [4]:
def score(picks, rows=ALL):
    acc = {}
    for kind in ["분명", "경계", "새"]:
        pairs = [(p, g) for p, (g, k, _) in zip(picks, rows) if k == kind]
        if pairs:
            acc[kind] = sum(p == g for p, g in pairs) / len(pairs)
    return acc


def confusion(picks, rows=ALL):
    cm = Counter((g, p) for p, (g, _, _) in zip(picks, rows))
    print(f"{'정답/선택':12s}" + "".join(f"{n:>9s}" for n in NAMES))
    for g in NAMES:
        print(f"{g:12s}" + "".join(f"{cm[(g, p)]:9d}" for p in NAMES))


def show(label, picks):
    acc = score(picks)
    print(f"{label:14s}" + "  ".join(f"{k} {v:4.0%}" for k, v in acc.items()))


RESULTS = {}

### 1. 규칙 라우터

세는 말(몇 편, 가장 많이, 나열 등)이 있으면 counter, 문서 제목이 들어 있으면 reader, 둘 다 없으면 refuser 다.
`DEV`를 보면서 낱말을 하나씩 더해 만든 규칙이다.

In [5]:
COUNT_WORDS = re.compile(r"몇 편|몇 개|몇 번|문서 수|가장 (많이|자주)|더 많|센다|나열|제목을 알려|어느 쪽")
NAMES_IN_TITLES = [t.replace("_", " ").split(" (")[0] for t in TITLES]


def rule_route(q):
    if COUNT_WORDS.search(q):
        return "counter"
    if any(n in q for n in NAMES_IN_TITLES):
        return "reader"
    return "refuser"


t0 = time.perf_counter()
rule_picks = [rule_route(q) for _, _, q in ALL]
RESULTS["규칙"] = {"picks": rule_picks, "초": (time.perf_counter() - t0) / len(ALL), "비용": 0.0}
show("규칙", rule_picks)

규칙            분명  62%  경계  56%  새  33%


`DEV`에서는 높고 `HELD`에서는 떨어진다. 새 질문에서 틀린 것을 본다.

In [6]:
for p, (g, k, q) in zip(rule_picks, ALL):
    if k == "새" and p != g:
        print(f"  {g:8s} -> {p:8s} {q}")

  counter  -> refuser  광복군은 문서 몇 군데에서 언급되는가
  counter  -> refuser  임시의정원 언급이 제일 많은 글은
  counter  -> refuser  조선민족혁명당 이야기가 나오는 글이 열 편을 넘나
  counter  -> refuser  워싱턴이라는 지명이 들어간 문서를 모두 찾아 달라
  counter  -> refuser  신한청년당과 국민대표회의 가운데 더 널리 언급되는 쪽은
  counter  -> refuser  이승만이라는 이름은 전체에서 몇 차례 나오나
  counter  -> refuser  대한매일신보를 언급하는 글의 목록
  reader   -> refuser  강우규가 1919년에 무슨 일을 했는지 알려 달라
  reader   -> refuser  박은식의 대표 저서는 무엇인가
  reader   -> refuser  독립공채표가 무엇이었는지 설명해 달라
  reader   -> refuser  백의사는 어떤 조직인가
  reader   -> refuser  안재홍은 해방 후 어떤 활동을 했나
  reader   -> refuser  남궁억이 세운 학교의 이름은
  reader   -> refuser  필라델피아 한인자유대회의 결의 내용은


"몇 군데", "제일 많은", "목록" 처럼 규칙에 없는 말로 물으면 놓친다. 규칙은 만든 사람이 본 질문에 맞춰진다.
그래서 라우터를 고칠 때 본 질문으로 정답률을 재면 실제보다 높게 나온다. `HELD`를 따로 둔 이유다.

### 2. 임베딩 라우터

담당자마다 예시 질문 다섯 개를 두고, 질문과 가장 가까운 예시의 담당자를 고른다. 예시는 `DEV`나 `HELD`와 겹치지 않게 따로 썼다.
1위와 2위 담당자의 **유사도 차이(margin)**를 같이 돌려준다. 뒤에서 확신의 척도로 쓴다.

In [7]:
EXAMPLES = {
    "counter": ["대한인국민회는 몇 편의 문서에 나오는가", "임시의정원이 가장 많이 나오는 문서는",
                "만주가 언급된 문서를 센다", "권업회가 나오는 문서 목록",
                "블라디보스토크와 하얼빈 중 어느 쪽이 더 많이 나오나"],
    "reader": ["안창호 문서는 그를 어떻게 소개하는가", "이상설 문서의 첫 문단을 요약한다",
               "박열 문서에 나오는 사건의 날짜는", "국제공산당 자금사건 문서는 무엇을 다루는가",
               "주시경 문서에 따르면 그의 업적은"],
    "refuser": ["내일 부산 날씨", "자바스크립트 비동기 처리 설명", "환율 계산해 줘",
                "여행 일정 짜 줘", "주식 추천해 줘"],
}


def embed(texts):
    r = keep_embed(client.embeddings.create(model=EMBEDDING, input=texts))
    v = np.array([d.embedding for d in r.data])
    return v / np.linalg.norm(v, axis=1, keepdims=True)


EX = {k: embed(v) for k, v in EXAMPLES.items()}


def embed_route(q):
    t0 = time.perf_counter()
    v = embed([q])[0]
    sims = sorted(((float((E @ v).max()), k) for k, E in EX.items()), reverse=True)
    return sims[0][1], sims[0][0] - sims[1][0], time.perf_counter() - t0

실제 서비스처럼 질문을 하나씩 임베딩한다.

In [8]:
mark = len(USAGE)
with ThreadPoolExecutor(4) as ex:
    emb_out = list(ex.map(embed_route, [q for _, _, q in ALL]))
RESULTS["임베딩"] = {"picks": [a for a, _, _ in emb_out], "conf": [m for _, m, _ in emb_out],
                    "초": np.mean([s for _, _, s in emb_out]), "비용": dollars(USAGE[mark:])}
show("임베딩", RESULTS["임베딩"]["picks"])
confusion(RESULTS["임베딩"]["picks"])

임베딩           분명  75%  경계  62%  새  67%
정답/선택         counter   reader  refuser
counter            19        1        0
reader              8       13        1
refuser             5        4       10


예시가 다섯 개뿐이라 **질문의 주제어**에 끌린다. 김구나 임시정부가 들어간 범위 밖 질문이 reader 나 counter 로 간다.

### 3. 모델 라우터

`11`의 지시문 그대로 담당자 이름 하나만 JSON 으로 받는다. 응답에 토큰별 로그 확률을 같이 달라고 해서, 담당자 이름 첫 토큰의 확률을 확신으로 쓴다.
스키마가 이름 셋을 enum 으로 묶으므로 그 토큰이 곧 선택이다.

In [ ]:
GUIDE = ("질문을 셋 중 하나로 보낸다. 문서 수나 빈도를 세는 질문은 counter,"
         " 특정 문서의 내용을 묻는 질문은 reader,"
         " 이 코퍼스(대한민국 임시정부 문서 50편) 밖의 질문은 refuser.")
SCHEMA = {"type": "json_schema", "name": "route", "strict": True,
          "schema": {"type": "object", "properties": {"agent": {"type": "string", "enum": NAMES}},
                     "required": ["agent"], "additionalProperties": False}}


def llm_route(model, q, guide=GUIDE):
    t0 = time.perf_counter()
    r = keep(client.responses.create(model=model, store=False, instructions=guide, input=q,
                                     text={"format": SCHEMA}, top_logprobs=5,   # 다음에 올 토큰들 중 확률 높은것 5개
                                     include=["message.output_text.logprobs"])) # 토큰에 대한 확률을 얼마나 confindent하게 보고있는가? (확신정도)
    agent = json.loads(r.output_text)["agent"]
    lp = r.output[-1].content[0].logprobs
    i = next(k for k, t in enumerate(lp) if t.token.startswith("agent")) + 2  # agent, ":" 다음 토큰
    return agent, math.exp(lp[i].logprob), time.perf_counter() - t0


agent, p, sec = llm_route(LOWER, "신채호는 어떤 사람인가")
print(agent, f"p={p:.3f}", f"{sec:.2f}초")

refuser p=0.997 1.06초


하위 모델과 상위 모델로 예순한 개를 모두 보낸다.

In [ ]:
def run_llm(label, model, guide=GUIDE):
    mark = len(USAGE)
    with ThreadPoolExecutor(4) as ex:
        out = list(ex.map(lambda row: llm_route(model, row[2], guide), ALL))
    RESULTS[label] = {"picks": [a for a, _, _ in out], "conf": [p for _, p, _ in out],
                      "초": np.mean([s for _, _, s in out]), "비용": dollars(USAGE[mark:])}
    show(label, RESULTS[label]["picks"])


run_llm("하위", LOWER)
run_llm("상위", MODEL)  # 상위모델이 단어에 대한 집착?이 더 커서 성능이 내려갈 수도 있음

하위            분명 100%  경계  94%  새  95%
상위            분명 100%  경계  94%  새  90%


두 모델의 혼동 행렬을 본다.

In [11]:
for label in ["하위", "상위"]:
    print(label)
    confusion(RESULTS[label]["picks"])

하위
정답/선택         counter   reader  refuser
counter            20        0        0
reader              0       20        2
refuser             0        0       19
상위
정답/선택         counter   reader  refuser
counter            20        0        0
reader              0       19        3
refuser             0        0       19


틀린 질문을 꺼내 본다.

In [12]:
for label in ["하위", "상위"]:
    for p, (g, k, q) in zip(RESULTS[label]["picks"], ALL):
        if p != g:
            print(f"{label}  {g:8s} -> {p:8s} [{k}] {q}")

하위  reader   -> refuser  [경계] 신채호는 어떤 사람인가
하위  reader   -> refuser  [새] 박은식의 대표 저서는 무엇인가
상위  reader   -> refuser  [경계] 신채호는 어떤 사람인가
상위  reader   -> refuser  [새] 박은식의 대표 저서는 무엇인가
상위  reader   -> refuser  [새] 안재홍은 해방 후 어떤 활동을 했나


오류의 상당수가 reader 질문을 refuser 로 보낸 것이다. "신채호는 어떤 사람인가" 는 문서 이름을 말하지 않는다. 라우터는 코퍼스에 신채호 문서가 있는지 모르므로 일반 지식 질문으로 본다.
상위 모델로 바꿔도 이 오류는 남는다. 모델이 똑똑해서 풀 문제가 아니라 라우터에게 주지 않은 정보의 문제다.

### 4. 라우터에게 범위를 보여 준다

지시문에 **문서 제목 50개**를 붙인다. 지시문이 길어지는 만큼 호출마다 입력 토큰이 는다.

In [14]:
GUIDE_TITLES = GUIDE + " 코퍼스의 문서 제목은 다음과 같다: " + ", ".join(t.replace("_", " ") for t in TITLES) + "."

run_llm("하위+제목", LOWER, GUIDE_TITLES)
run_llm("상위+제목", MODEL, GUIDE_TITLES)
for label in ["하위+제목", "상위+제목"]:
    for p, (g, k, q) in zip(RESULTS[label]["picks"], ALL):
        if p != g:
            print(f"{label}  {g:8s} -> {p:8s} [{k}] {q}")

하위+제목         분명 100%  경계  88%  새  81%
상위+제목         분명 100%  경계  81%  새  86%
하위+제목  reader   -> refuser  [경계] 신채호는 어떤 사람인가
하위+제목  reader   -> refuser  [경계] 한국친우회가 무엇인지 설명해 달라
하위+제목  counter  -> refuser  [새] 신한청년당과 국민대표회의 가운데 더 널리 언급되는 쪽은
하위+제목  counter  -> reader   [새] 대한매일신보를 언급하는 글의 목록
하위+제목  reader   -> refuser  [새] 박은식의 대표 저서는 무엇인가
하위+제목  reader   -> refuser  [새] 안재홍은 해방 후 어떤 활동을 했나
상위+제목  reader   -> refuser  [경계] 임시정부는 언제 세워졌는가
상위+제목  reader   -> refuser  [경계] 신채호는 어떤 사람인가
상위+제목  reader   -> refuser  [경계] 한국친우회가 무엇인지 설명해 달라
상위+제목  reader   -> refuser  [새] 강우규가 1919년에 무슨 일을 했는지 알려 달라
상위+제목  reader   -> refuser  [새] 박은식의 대표 저서는 무엇인가
상위+제목  reader   -> refuser  [새] 안재홍은 해방 후 어떤 활동을 했나


두 모델 모두 reader 를 refuser 로 보내는 오류는 줄어든다. 그런데 하위 모델은 대신 다른 쪽으로 틀리기 시작할 수 있다. 제목 목록을 본 뒤로 문서 이름이 든 목록 질문을 reader 로 보내는 식이다.
지시문을 고치면 오류가 없어지는 것이 아니라 옮겨 갈 수 있다. 그래서 고친 뒤에는 고치려던 질문만이 아니라 전체 셋을 다시 잰다.

### 5. 확신은 어디서 얻는가

대체 경로를 쓰려면 라우터가 덜 확실한 질문을 골라낼 수 있어야 한다. 후보는 셋이다. **모델이 스스로 말한 확신, 로그 확률, 임베딩의 margin** 이다.

먼저 모델에게 확신을 직접 묻는다. `11`의 `Route`에 확신 칸을 더했다.

In [15]:
class Route(BaseModel):
    agent: Literal["counter", "reader", "refuser"]
    confidence: Literal["높음", "낮음"]


def said_route(model, q):
    r = keep(client.responses.parse(model=model, store=False, text_format=Route, input=q,
                                    instructions=GUIDE + " 어느 쪽인지 확실하지 않으면 confidence 를 낮음으로 둔다."))
    return r.output_parsed


for label, model in [("하위", LOWER), ("상위", MODEL)]:
    with ThreadPoolExecutor(4) as ex:
        out = list(ex.map(lambda row: said_route(model, row[2]), ALL))
    rows = [(o.agent == g, o.confidence) for o, (g, _, _) in zip(out, ALL)]
    for c in ["높음", "낮음"]:
        sel = [ok for ok, cc in rows if cc == c]
        print(f"{label}  {c}  {len(sel):2d}개  그중 오답 {sum(not ok for ok in sel)}")

하위  높음   2개  그중 오답 0
하위  낮음  59개  그중 오답 7
상위  높음  58개  그중 오답 3
상위  낮음   3개  그중 오답 0


스스로 말한 확신은 모델의 버릇을 따른다. 한 모델은 거의 다 낮다고 하고 다른 모델은 거의 다 높다고 한다. 어느 쪽이든 오답을 골라내지 못한다.

로그 확률과 margin 은 문턱을 정해 자른다. **문턱 아래는 대체 경로로 보내고, 위는 자동으로 보낸다**. 문턱을 올리면 자동으로 처리하는 질문은 줄고 그 정확도는 오른다.

In [16]:
def coverage(label, thresholds):
    picks, conf = RESULTS[label]["picks"], RESULTS[label]["conf"]
    ok = [p == g for p, (g, _, _) in zip(picks, ALL)]
    for th in thresholds:
        kept = [o for o, c in zip(ok, conf) if c >= th]
        print(f"{label:9s} 문턱 {th:<6} 자동 {len(kept):2d}/{len(ok)}  정확 {sum(kept) / max(1, len(kept)):4.0%}"
              f"  대체 경로로 간 오답 {sum(not o for o, c in zip(ok, conf) if c < th)}/{ok.count(False)}")


coverage("임베딩", [0, 0.02, 0.04])
coverage("하위", [0, 0.9, 0.99])
coverage("상위", [0, 0.9, 0.99])

임베딩       문턱 0      자동 61/61  정확  69%  대체 경로로 간 오답 0/19
임베딩       문턱 0.02   자동 48/61  정확  75%  대체 경로로 간 오답 7/19
임베딩       문턱 0.04   자동 35/61  정확  89%  대체 경로로 간 오답 15/19
하위        문턱 0      자동 61/61  정확  97%  대체 경로로 간 오답 0/2
하위        문턱 0.9    자동 58/61  정확  97%  대체 경로로 간 오답 0/2
하위        문턱 0.99   자동 52/61  정확  96%  대체 경로로 간 오답 0/2
상위        문턱 0      자동 61/61  정확  95%  대체 경로로 간 오답 0/3
상위        문턱 0.9    자동 59/61  정확  95%  대체 경로로 간 오답 0/3
상위        문턱 0.99   자동 55/61  정확  96%  대체 경로로 간 오답 1/3


임베딩의 margin 은 문턱을 올리면 오답을 많이 걸러내지만 자동으로 처리하는 질문도 크게 준다.
로그 확률이 오답을 얼마나 골라내는지는 모델과 실행마다 다르다. 오답의 확률이 거의 1 이면 어떤 문턱으로도 걸러지지 않는다. 범위를 모르는 오류가 대개 이렇다. 라우터는 모르는 것을 모른다.

### 6. 대체 경로

대체 경로는 둘이다. **사용자에게 되묻거나, 도구를 모두 가진 일반 에이전트**에게 보낸다.
되묻기는 비용이 들지 않지만 사용자에게 한 턴을 더 요구한다. 일반 에이전트는 사용자를 기다리게 하지 않지만 도구와 지시가 넓어서 호출이 비싸다.

먼저 하위 모델 라우터에서 확률 0.9 아래로 떨어진 질문이 실제로 틀린 것이었는지 센다.

In [17]:
low = [(p, row) for p, c, row in zip(RESULTS["하위"]["picks"], RESULTS["하위"]["conf"], ALL) if c < 0.9]
print(f"낮은 확신 {len(low)}개, 그중 실제 오답 {sum(p != row[0] for p, row in low)}개")
for p, (g, k, q) in low:
    print(f"  {'오답' if p != g else '    '} {g:8s} -> {p:8s} {q}")

낮은 확신 3개, 그중 실제 오답 0개
       counter  -> counter  안중근이 언급된 문서들의 제목을 알려 달라
       reader   -> reader   국민대표회의 문서에 참가자가 몇 명이었다고 나오는가
       reader   -> reader   한국친우회가 무엇인지 설명해 달라


걸린 질문의 대부분이 맞게 라우팅된 것이면 대체 경로는 비용만 늘린다. 같은 질문을 라우터가 고른 전문가와 일반 에이전트에게 각각 보내 답과 비용을 본다.
루프는 `9`의 코어 블록이다. 일반 에이전트도 도구를 모두 가진 설정 하나다.

In [19]:
def call_tool(name, args, cfg, approve):
    if name in cfg["gated"] and not approve:
        return cfg["gate_error"]
    try:
        return json.dumps(cfg["impl"][name](**args), ensure_ascii=False)
    except Exception as e:
        return cfg["tool_error"].format(e=e)


def run(task, cfg=CONFIG, approve=False):
    messages = [{"role": "user", "content": task}]
    trace, t0, used = [], time.perf_counter(), 0
    for turn in range(1, cfg["max_turns"] + 1):
        r = keep(client.responses.create(model=cfg["model"], instructions=cfg["instructions"],
                                         input=messages, tools=cfg["tools"], store=False))
        used += r.usage.total_tokens
        calls = [i for i in r.output if i.type == "function_call"]
        trace.append({"턴": turn, "도구": [c.name for c in calls], "토큰": used,
                      "초": round(time.perf_counter() - t0, 1)})
        if not calls:
            return r.output_text, "완료", trace
        if used > cfg["max_tokens"]:
            return None, "토큰 예산", trace
        if time.perf_counter() - t0 > cfg["max_seconds"]:
            return None, "시간 예산", trace
        messages += [i.model_dump() for i in r.output]
        messages += [{"type": "function_call_output", "call_id": c.call_id,
                      "output": call_tool(c.name, json.loads(c.arguments), cfg, approve)}
                     for c in calls]
    return None, "반복 예산", trace


GENERAL = {**CONFIG, "instructions": ("이 코퍼스는 대한민국 임시정부 문서 50편이다. 세는 질문은 grep_docs, 문서 내용은 read_doc 으로 확인하고"
                                       " 확인한 것만 답한다. 코퍼스 밖 질문에는 답할 수 없다고 말한다.")}

cost_s, cost_g = 0.0, 0.0
for picked, (gold, _, q) in low[:4]:
    m = len(USAGE)
    a = run(q, AGENTS[picked])[0]
    cost_s += dollars(USAGE[m:])
    m = len(USAGE)
    b = run(q, GENERAL)[0]
    cost_g += dollars(USAGE[m:])
    print(f"{q}  (정답 {gold}, 라우터 {picked})")
    print(f"  전문가 | {' '.join(str(a).split())[:70]}")
    print(f"  일반   | {' '.join(str(b).split())[:70]}")
n = max(1, min(4, len(low)))
print(f"\n질문당 비용  전문가 ${cost_s / n:.4f}   일반 ${cost_g / n:.4f}")

안중근이 언급된 문서들의 제목을 알려 달라  (정답 counter, 라우터 counter)
  전문가 | 0
  일반   | 코퍼스에서 “안중근”을 검색해 보았지만, 언급된 문서는 확인되지 않았습니다.
국민대표회의 문서에 참가자가 몇 명이었다고 나오는가  (정답 reader, 라우터 reader)
  전문가 | 제가 가진 문서 목록에는 **「국민대표회의」**라는 제목의 문서가 없습니다. 정확한 문서 제목을 다시 알려주시면 그 문서를 읽
  일반   | 확인 가능한 코퍼스에서 “국민대표회의”가 들어 있는 문서를 찾지 못했습니다. 따라서 이 코퍼스만으로는 참가자가 몇 명이었다고 
한국친우회가 무엇인지 설명해 달라  (정답 reader, 라우터 reader)
  전문가 | 제가 가진 문서 목록에는 **「한국친우회」**라는 제목의 문서가 없습니다. 혹시 다음 중 하나를 뜻하셨나요? - **친우회**
  일반   | 제게 주어진 대한민국 임시정부 문서 50편 코퍼스에서는 **“한국친우회”**라는 낱말이 확인되지 않습니다. 따라서 이 코퍼스에

질문당 비용  전문가 $0.0017   일반 $0.0021


확신으로 거르는 방식은 확신을 가지고 틀린 질문을 놓친다. 이 셋에서는 그런 오답이 대개 refuser 쪽이다. 답할 수 있는 질문을 refuser 로 보내면 refuser 는 답할 수 없다고 하거나, 지시를 어기고 도구 없이 기억으로 답한다. 어느 쪽이든 코퍼스로 확인한 답이 아니고, 겉으로는 정상처럼 보인다.

그래서 확신 대신 담당자로 거른다. **라우터가 refuser 를 고른 질문만 일반 에이전트에게 한 번 더 보낸다.** 거절은 틀려도 티가 나지 않지만 다시 확인하는 비용은 범위 밖 질문의 수만큼만 든다.
일반 에이전트가 실제로 답했는지는 하위 모델이 판정한다.

In [21]:
class Handled(BaseModel):
    답함: bool


def answered(q, a):
    return keep(client.responses.parse(
        model=LOWER, store=False, text_format=Handled,
        input=f"질문 {q}\n답 {a}\n\n답이 질문에 실제로 답했으면 참, 답할 수 없다고 했거나 확인하지 못했다고 했으면 거짓.")
    ).output_parsed.답함


to_refuser = [row for p, row in zip(RESULTS["하위"]["picks"], ALL) if p == "refuser"]
mark = len(USAGE)
with ThreadPoolExecutor(4) as ex:
    replies = list(ex.map(lambda row: run(row[2], GENERAL)[0], to_refuser))
with ThreadPoolExecutor(4) as ex:
    flags = list(ex.map(lambda x: answered(x[0][2], x[1]), zip(to_refuser, replies)))

rescued = [row[2] for row, f in zip(to_refuser, flags) if f and row[0] != "refuser"]
missed = [row[2] for row, f in zip(to_refuser, flags) if not f and row[0] != "refuser"]
invented = [row[2] for row, f in zip(to_refuser, flags) if f and row[0] == "refuser"]
print(f"refuser 로 간 질문 {len(to_refuser)}개, 재확인 비용 ${dollars(USAGE[mark:]):.4f}")
print(f"  잘못 거절될 뻔한 것을 답함 {len(rescued)}  {rescued}")
print(f"  잘못 거절된 채 남음      {len(missed)}  {missed}")
print(f"  범위 밖인데 답함        {len(invented)}  {invented}")

refuser 로 간 질문 21개, 재확인 비용 $0.0352
  잘못 거절될 뻔한 것을 답함 0  []
  잘못 거절된 채 남음      2  ['신채호는 어떤 사람인가', '박은식의 대표 저서는 무엇인가']
  범위 밖인데 답함        2  ['영어 이메일 초안을 써 달라', '이 문장을 영어로 번역해 달라: 안녕하세요']


잘못 간 질문은 일반 에이전트가 도구로 확인하고 답한다. 대신 범위 밖 질문에 일반 에이전트가 답을 지어내면 새 오류가 생긴다. 마지막 줄이 그 수다.
이 재확인은 라우터를 고치는 것이 아니라 한 방향의 실패를 다른 단계에서 받아 내는 것이다. 어느 방향의 실패가 더 비싼지에 따라 어디에 두 번째 확인을 둘지 정한다.
</br>결정적이지 않으며 모델 업데이트에 따라서도 달라질 수 있음 => 지속적인 모니터링과 골든데이터셋을 바탕으로 한 평가가 중요

### 나란히 둔다

`HELD` 정답률이 실제에 가까운 숫자다. 비용은 질문 천 개를 보낼 때로 환산한다.

In [22]:
for label, r in RESULTS.items():
    acc = score(r["picks"])
    print(f"{label:9s}  분명 {acc['분명']:4.0%}  경계 {acc['경계']:4.0%}  새 {acc['새']:4.0%}"
          f"   ${r['비용'] / len(ALL) * 1000:6.3f}/1000건   {r['초'] * 1000:6.0f}ms")

규칙         분명  62%  경계  56%  새  33%   $ 0.000/1000건        0ms
임베딩        분명  75%  경계  62%  새  67%   $ 0.000/1000건      185ms
하위         분명 100%  경계  94%  새  95%   $ 0.143/1000건      834ms
상위         분명 100%  경계  94%  새  90%   $ 0.469/1000건     1146ms
하위+제목      분명 100%  경계  88%  새  81%   $ 0.158/1000건      842ms
상위+제목      분명 100%  경계  81%  새  86%   $ 0.502/1000건     1081ms


### 라우터 고르기

규칙 라우터는 공짜이고 빠르지만 만든 사람이 본 말투에만 맞는다. 본 질문으로 잰 정답률은 믿지 않는다.
임베딩 라우터는 예시가 적으면 주제어에 끌린다. margin 은 쓸 만한 확신 척도라서 다른 라우터 앞에 거르는 단계로 둘 만하다.

모델 라우터의 오류는 모델의 크기보다 라우터가 무엇을 아는지에 달렸다. 상위 모델로 바꿔도 범위를 모르는 오류는 그대로 남는다.
담당 범위를 지시문에 넣으면 그 오류는 준다. 다만 하위 모델에서는 오류가 다른 방향으로 옮겨 갈 수 있으니 고친 뒤 전체 셋을 다시 잰다.

확신은 모델에게 묻지 않는다. 로그 확률이나 margin 처럼 출력 밖에서 재도, 확신을 가지고 틀리는 질문은 남는다.
그래서 대체 경로는 확신만이 아니라 실패의 방향으로도 건다. 거절처럼 틀려도 티가 나지 않는 선택에 두 번째 확인을 둔다.

### 비용 확인

In [ ]:
print(f"호출 {len(USAGE)}회")
print(f"입력 {sum(u.input_tokens for _, u in USAGE):,} 토큰   출력 {sum(u.output_tokens for _, u in USAGE):,} 토큰")
print(f"누적 ${dollars(USAGE):.4f}")